# Walkthrough: one utterance through the six conditions

This offline example uses only artifacts tracked in the repository. It makes no API calls and does not require the TalkMoves source workbook. It follows one utterance through the six condition prompts and the preserved raw model responses to its final label.

The utterance comes from the main run (`runs/main-2026-09-24/`). The notebook compares the manual sections of the six prompts. It then shows the target line with its context window and the raw API responses. For each response it derives the parser category and the final label per condition. It reuses the repository's own loader and parser functions and adds no parsing logic of its own.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()                      # this notebook lives in examples/
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from build_inputs import CONDITIONS, CONTEXT_HEADER, MANUAL_HEADER, TARGET_MARKER
from parse_attempts import aggregate, attempts_files, labels_from_records, read_final_labels, read_jsonl_strict

RUN_DIR = REPO_ROOT / "runs" / "main-2026-09-24"
UTTERANCE_ID = 24222

pd.set_option("display.max_colwidth", 60)
print("repository:", REPO_ROOT.name, "| run directory:", RUN_DIR.relative_to(REPO_ROOT))

repository: coding-manual-ablation | run directory: runs/main-2026-09-24


## 1. The six prompts and their manual sections

`prompts.jsonl` holds one record per (utterance_id, condition) with the full prompt text. Each prompt is four sections joined by blank lines: task instruction, coding manual (absent in `names_only`), context block, output instruction. The table below compares the manual section of each condition with the baseline manual line by line, because placeholder replacement keeps the line structure and replaces only the text of selected lines.

In [2]:
prompts = {r["condition"]: r["prompt"] for r in read_jsonl_strict(RUN_DIR / "prompts.jsonl")
           if r["utterance_id"] == UTTERANCE_ID}
assert set(prompts) == set(CONDITIONS), sorted(prompts)


def sections(prompt: str) -> dict[str, str]:
    """Split one prompt into its named sections using the headers defined in build_inputs."""
    out = {}
    body = prompt
    if MANUAL_HEADER in body:
        before, after = body.split("\n\n" + MANUAL_HEADER + "\n", 1)
        out["task"] = before
        manual, body = after.split("\n\n" + CONTEXT_HEADER + "\n", 1)
        out["manual"] = manual
    else:
        out["task"], body = body.split("\n\n" + CONTEXT_HEADER + "\n", 1)
    out["context"], out["output"] = body.split("\n\nOutput:", 1)
    out["output"] = "Output:" + out["output"]
    return out


parts = {c: sections(p) for c, p in prompts.items()}
base_lines = parts["baseline"]["manual"].split("\n")
heading = re.compile(r"^1(\.\d+)? ")          # section headings of the manual transcription (1, 1.1, ... 1.6)

rows = []
for c in CONDITIONS:
    s = parts[c]
    if "manual" not in s:
        rows.append({"condition": c, "prompt chars": len(prompts[c]), "manual lines": 0,
                     "lines differing from baseline": None, "placeholder chars ('%')": 0, "sections touched": "(no manual)"})
        continue
    lines = s["manual"].split("\n")
    assert len(lines) == len(base_lines), (c, len(lines), len(base_lines))
    changed = [i for i, (a, b) in enumerate(zip(base_lines, lines)) if a != b]
    touched = []
    for i in changed:
        h = next(base_lines[j] for j in range(i, -1, -1) if heading.match(base_lines[j]))
        if h.split(" ")[0] not in touched:
            touched.append(h.split(" ")[0])
    rows.append({"condition": c, "prompt chars": len(prompts[c]), "manual lines": len(lines),
                 "lines differing from baseline": len(changed), "placeholder chars ('%')": s["manual"].count("%"),
                 "sections touched": ", ".join(touched) if touched else "none"})
manual_table = pd.DataFrame(rows).set_index("condition")
manual_table["lines differing from baseline"] = manual_table["lines differing from baseline"].astype("Int64")
manual_table

,prompt chars,manual lines,lines differing from baseline,placeholder chars ('%'),sections touched
condition,,,,,
baseline,9298,132,0,0,none
definition_replacement,8246,132,6,314,"1.3, 1.4, 1.5"
example_replacement,8211,132,72,602,"1.3, 1.4, 1.5"
exclusion_rule_replacement,8770,132,11,171,1.6
negative_control,9040,132,1,79,1.1
names_only,1320,0,<NA>,0,(no manual)


In [3]:
# One replaced line per condition, baseline text against condition text (first differing line only).
for c in ["definition_replacement", "example_replacement", "exclusion_rule_replacement", "negative_control"]:
    lines = parts[c]["manual"].split("\n")
    i = next(i for i, (a, b) in enumerate(zip(base_lines, lines)) if a != b)
    print(f"{c}  (manual line {i + 1})")
    print("  baseline :", base_lines[i][:110] + ("..." if len(base_lines[i]) > 110 else ""))
    print("  condition:", lines[i][:110] + ("..." if len(lines[i]) > 110 else ""))
    print()

definition_replacement  (manual line 20)
  baseline : Teacher prompts for students to be active listeners are coded as Keeping Everyone Together. This code is used ...
  condition: % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % ...

example_replacement  (manual line 22)
  baseline : ■ So x equals five dollars, right?
  condition: ■ % % % % % % % %

exclusion_rule_replacement  (manual line 122)
  baseline : ➢ Teacher utterances that neither ask for a student contribution nor repeat all or part of a student contribut...
  condition: ➢ % % % % % % % % % % % % % % % % % % % %

negative_control  (manual line 3)
  baseline : The talk moves described in this manual are grounded in accountable talk theory (Michaels, et al., 2008; 2010)...
  condition: % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % % ...



## 2. Target line and context window

The context block is the same in all six prompts. The target row carries the `[TARGET]` marker; the other rows are the context window (up to seven rows on each side, same transcript).

In [4]:
context = parts["baseline"]["context"]
assert all(parts[c]["context"] == context for c in CONDITIONS)
ctx_lines = context.split("\n")
target = next(l for l in ctx_lines if l.startswith(TARGET_MARKER))
print("target  :", target[len(TARGET_MARKER):])
print("window  :", len(ctx_lines), "rows,", ctx_lines.index(target), "before the target,", len(ctx_lines) - ctx_lines.index(target) - 1, "after")
print()
print(context)

target  : T: Well then write it down From one of the two  thats a special case
window  : 15 rows, 7 before the target, 7 after

T: Corine I will correct it for you by tomorrow all right
T: I dont have it yet
S: Mhm yes
T: Do it in your notebook Shane so that youre able to make a clean entry of the solution here afterwards
T: Yes and now now you have to think about it what does it mean that is zero
T: Under what circumstances can
S: Actually it should be a zero there too
[TARGET] T: Well then write it down From one of the two  thats a special case
S: Maam is that right
T: I dont know if thats a beneficial beginning
S: Cant you do that
T: Do it first in your notebook
S: Yees
T: Hm then it gets problematic right because youve got X squared and X
S: How con you solve this


## 3. Raw responses

`attempts_pass1.jsonl` records every API call. An `attempt_completed` record stores the HTTP body returned by the API in `raw_response`. Three repeats were planned per (utterance, condition); the tie-break file `attempts_pass2.jsonl` is read as well in case this utterance needed one.

In [5]:
records = [r for path in attempts_files(RUN_DIR) for r in read_jsonl_strict(path)]
completed = [r for r in records if r.get("event") == "attempt_completed" and r["utterance_id"] == UTTERANCE_ID]
completed.sort(key=lambda r: (r["pass"], CONDITIONS.index(r["condition"]), r["repeat"], r["attempt"]))

resp = pd.DataFrame([{
    "pass": r["pass"], "condition": r["condition"], "repeat": r["repeat"], "attempt": r["attempt"],
    "http_status": r["http_status"], "outcome": r["outcome"], "finish_reason": r["finish_reason"],
    "response_model": r["response_model"],
    "message content": json.loads(r["raw_response"])["choices"][0]["message"]["content"],
} for r in completed])
resp

,pass,condition,repeat,attempt,http_status,outcome,finish_reason,response_model,message content
0,1,baseline,1,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Revoicing""}"
1,1,baseline,2,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Not coded""}"
2,1,baseline,3,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Not coded""}"
3,1,definition_replacement,1,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Not coded""}"
4,1,definition_replacement,2,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Not coded""}"
5,1,definition_replacement,3,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Not coded""}"
6,1,example_replacement,1,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Revoicing""}"
7,1,example_replacement,2,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Revoicing""}"
8,1,example_replacement,3,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Revoicing""}"
9,1,exclusion_rule_replacement,1,1,200,success,stop,gpt-5.5-2026-04-23,"{""category"":""Keeping Everyone Together""}"


In [6]:
# One complete raw response, as stored (baseline, repeat 1).
first = next(r for r in completed if r["condition"] == "baseline" and r["repeat"] == 1)
print(first["raw_response"])

{
  "id": "chatcmpl-ERWddwZWuP9v2CUsZCANf7ci2Wvdl",
  "object": "chat.completion",
  "created": 1790229025,
  "model": "gpt-5.5-2026-04-23",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "{\"category\":\"Revoicing\"}",
        "refusal": null,
        "annotations": []
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 2176,
    "completion_tokens": 15,
    "total_tokens": 2191,
    "prompt_tokens_details": {
      "cached_tokens": 1792,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0,
      "audio_tokens": 0,
      "accepted_prediction_tokens": 0,
      "rejected_prediction_tokens": 0
    }
  },
  "service_tier": "default",
  "system_fingerprint": null
}



## 4. Parser category and final label

`parse_attempts.labels_from_records` re-derives the category of every completed attempt from `raw_response` with the repository's validation function. `parse_attempts.aggregate` applies the aggregation rule of Decision Log 5.4 to those rows (majority over the valid repeats; a tie triggers a tie-break call). The last column is the value stored in `final_labels.csv`.

In [7]:
label_rows = [r for r in labels_from_records(records) if r["utterance_id"] == UTTERANCE_ID]
agg = aggregate(label_rows)
stored = {r["condition"]: r for r in read_final_labels(RUN_DIR / "final_labels.csv") if int(r["utterance_id"]) == UTTERANCE_ID}

table = []
for c in CONDITIONS:
    reps = {r["repeat"]: (r["category"] if r["valid"] else f"invalid") for r in label_rows if r["condition"] == c}
    a = agg[(UTTERANCE_ID, c)]
    table.append({
        "condition": c,
        **{f"repeat {k}": reps.get(k, "") for k in sorted(reps)},
        "valid repeats": a["valid_repeats"],
        "counts": a["counts"],
        "parser final_label": a["final_label"],
        "status": a["status"],
        "final_labels.csv": stored[c]["final_label"],
        "match": a["final_label"] == stored[c]["final_label"] and a["status"] == stored[c]["status"],
    })
pd.DataFrame(table).set_index("condition")

,repeat 1,repeat 2,repeat 3,valid repeats,counts,parser final_label,status,final_labels.csv,match
condition,,,,,,,,,
baseline,Revoicing,Not coded,Not coded,3,"{'Revoicing': 1, 'Not coded': 2}",Not coded,resolved,Not coded,True
definition_replacement,Not coded,Not coded,Not coded,3,{'Not coded': 3},Not coded,resolved,Not coded,True
example_replacement,Revoicing,Revoicing,Revoicing,3,{'Revoicing': 3},Revoicing,resolved,Revoicing,True
exclusion_rule_replacement,Keeping Everyone Together,Revoicing,Keeping Everyone Together,3,"{'Keeping Everyone Together': 2, 'Revoicing': 1}",Keeping Everyone Together,resolved,Keeping Everyone Together,True
negative_control,Revoicing,Revoicing,Not coded,3,"{'Revoicing': 2, 'Not coded': 1}",Revoicing,resolved,Revoicing,True
names_only,Pressing for Accuracy,Pressing for Accuracy,Pressing for Accuracy,3,{'Pressing for Accuracy': 3},Pressing for Accuracy,resolved,Pressing for Accuracy,True


Reading the table: where the three repeats agree, the final label is that category. Where they disagree (`baseline`, `exclusion_rule_replacement`, `negative_control` for this utterance), the final label is the category named by two of the three valid repeats, which is the majority rule of Decision Log 5.4.2. No repeat of this utterance was invalid and no tie occurred, so no tie-break call was made for it. The `match` column confirms that the parser output recomputed here equals the stored `final_labels.csv` row for every condition.

Cohen's κ is not computed in this notebook. It needs the human labels, which are derived from the source workbook and are not tracked in the repository; see `docs/reproduction.md`.